# Comparison with Other Implementations

This page compares the flow cartogram of carto-flow with other implementations on the US census data (Albers equal-area projection):

| Tool | Method |
|---|---|
| carto-flow | diffusion (Gastner and Newman), FFT solver |
| go-cart (`cartogram-cpp`) | diffusion (Gastner and Newman), C++ |
| R `cartogramR`, `gsm` | diffusion, fast solver of Gastner, Seguy and More (2018) |
| R `cartogramR`, `gn` | diffusion (Gastner and Newman) |
| R `cartogram` (`cartogram_cont`) | Dougenik, Chrisman and Niemeyer (1985) |
| Python `cartogram` | Dougenik, Chrisman and Niemeyer (1985) |

## How the comparison is made

A tool does not decide when it is accurate enough. The iterative tools run with a stopping tolerance that cannot be reached and an increasing iteration cap, so each run stops at its cap. go-cart has no iteration cap; it runs with a ladder of maximum area errors, which is also its stopping rule. All tools run at their default grid. Every run is a separate run from the input geometries.

The accuracy of every output is measured here, from the output geometries and the data values, the same way for every tool:

- **area error**: the relative deviation of each region's share of the total area from its share of the total value, as maximum and mean over the regions;
- **validity**: the number of output regions that are not valid polygons.

The plots show the achieved error against the median time of each run. The line is the best achievable error at a given time (a setting is on the line if no other setting of the same tool is both faster and more accurate); the faint points are all settings. The table reads the shortest time that reaches an error target off the data. Times are only comparable within one run and one machine. The Dougenik-type tools are limited to fewer iterations on the larger datasets because their cost per iteration grows with the number of regions times the number of vertices; "not reached" then means that the largest setting run did not reach the target.

The comparison is reproducible with the Docker image in `benchmarks/comparison/`.

In [ ]:
import json
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from IPython.display import Markdown

_here = os.getcwd()
_dir = next(d for d in (_here, os.path.join(_here, "docs", "performance")) if os.path.isdir(d))
_TARGETS = ["5%", "1%", "0.5%"]
_FLOOR = 1e-4  # errors below this (%) are drawn at the floor on the log axis


def _load(dataset):
    path = os.path.join(_dir, f"comparison_{dataset}.json")
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return None
    with open(path) as f:
        return json.load(f)


def _frontier(runs, key):
    """Runs that no other run beats on both time and error, ordered by time."""
    best, out = np.inf, []
    for r in sorted(runs, key=lambda r: r["time_s"]):
        if r[key] < best:
            best = r[key]
            out.append(r)
    return out


def _curves(data):
    tools = list(data["tools"])
    palette = dict(zip(tools, sns.color_palette("tab10", len(tools)), strict=False))
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True)
    for ax, key, label in zip(
        axes,
        ["max_area_error_pct", "mean_area_error_pct"],
        ["maximum area error (%)", "mean area error (%)"],
        strict=False,
    ):
        for tool, entry in data["tools"].items():
            runs = entry["runs"]
            t = [r["time_s"] for r in runs]
            e = [max(r[key], _FLOOR) for r in runs]
            ax.scatter(t, e, s=14, color=palette[tool], alpha=0.35)
            fr = _frontier(runs, key)
            ax.plot(
                [r["time_s"] for r in fr],
                [max(r[key], _FLOOR) for r in fr],
                "-o",
                ms=4,
                color=palette[tool],
                label=tool,
            )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("time (s)")
        ax.set_ylabel(label)
        ax.grid(alpha=0.3)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=4, fontsize=8, frameon=False)
    plt.tight_layout(rect=(0, 0.1, 1, 1))
    plt.show()


def _summary(data):
    md = "| tool | " + " | ".join(f"time to {t} (s)" for t in _TARGETS) + " | invalid regions (max over runs) |\n"
    md += "|" + "|".join(" --- " for _ in range(len(_TARGETS) + 2)) + "|\n"
    for tool, entry in data["tools"].items():
        times = [entry["time_to_target_s"][t] for t in _TARGETS]
        cells = ["not reached" if v is None else f"{v:.2f}" for v in times]
        md += f"| {tool} | " + " | ".join(cells) + f" | {max(r['n_invalid'] for r in entry['runs'])} |\n"
    return Markdown(md)


def _examples(dataset, data):
    """One cartogram per tool, each from its fastest run within 1 % maximum area error (else its best)."""
    path = os.path.join(_dir, f"comparison_{dataset}.gpkg")
    if not os.path.exists(path):
        return
    panels = [("input", None)] + [
        (tool, entry["example"]) for tool, entry in data["tools"].items() if "example" in entry
    ]
    ncols = 3
    nrows = -(-len(panels) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 2.9 * nrows))
    lw = 0.3 if dataset == "states" else 0.1
    for ax in axes.ravel():
        ax.set_axis_off()
    for ax, (name, ex) in zip(axes.ravel(), panels, strict=False):
        gdf = gpd.read_file(path, layer=name)
        gdf.plot(
            ax=ax, column="value", cmap="viridis", norm=plt.matplotlib.colors.LogNorm(), edgecolor="white", linewidth=lw
        )
        if ex is None:
            title = "input"
        else:
            setting = f"{ex['max_iterations']} it." if "max_iterations" in ex else f"tol. {ex['tolerance']:g}"
            title = f"{name}\n{setting}, {ex['time_s']:.1f} s, max error {ex['max_area_error_pct']:.2g} %, {ex['n_invalid']} invalid"
        ax.set_title(title, fontsize=8)
    fig.tight_layout()
    plt.show()


def _section(dataset):
    data = _load(dataset)
    if data is None:
        display(Markdown(f"No comparison results for the {dataset} in `comparison_{dataset}.json` yet."))
        return
    display(
        Markdown(
            f"{data['n_regions']} regions, {data['repeats']} timed run(s) per setting, profile `{data['profile']}`."
        )
    )
    _curves(data)
    display(_summary(data))
    display(
        Markdown(
            "Example cartograms: each tool's fastest run within 1 % maximum area error, or its most accurate run if none reaches it. "
            "Colors show the value of each region."
        )
    )
    _examples(dataset, data)

## US states

49 regions (the 48 contiguous states and the District of Columbia).

In [ ]:
_section("states")

## Congressional districts

432 regions, geometries simplified at 1 km tolerance. carto-flow is also run with its two validity repair modes (`repair-crossings` and `repair-quadtree`).

In [ ]:
_section("districts")

## Counties

3,108 regions, geometries simplified at 1 km tolerance. Only the tools whose cost allows it are run.

In [ ]:
_section("counties")